# Week 3 — LightGBM: GOSS, EFB, and Leaf-wise Growth

> *Breaking the $O(n \cdot d)$ histogram ceiling. Two orthogonal speed-ups and one regularization-sensitive growth strategy.*

## Learning objectives

By the end of this notebook, you will be able to:

1. State the GOSS sampling rule and prove its split-gain estimator is unbiased.
2. Reduce Exclusive Feature Bundling to graph coloring and implement the greedy approximation.
3. Explain leaf-wise vs. level-wise tree growth and the regularization implications.
4. Benchmark XGBoost against LightGBM on training time, peak memory, and accuracy.
5. Diagnose and prevent the overfitting failure mode of leaf-wise growth.

## Outline

1. **GOSS** — Gradient-based One-Side Sampling
2. **Unbiasedness of GOSS** — the proof
3. **EFB** — Exclusive Feature Bundling as graph coloring
4. **Leaf-wise vs. level-wise growth**
5. **Head-to-head benchmark** — XGBoost vs. LightGBM
6. **Overfitting demonstration** under loose regularization


## 1. GOSS — Gradient-based One-Side Sampling

Histogram-construction cost of GBDT training is $O(n \cdot d)$ per iteration. Ke et al. (2017) observed that **instances with small $|g_i|$ are already well-fit and contribute little to the next split's gain**. We can downsample them and pay only a controlled variance cost.

### Algorithm

Given gradients $g_1, \dots, g_n$ and parameters $a, b \in (0, 1)$ with $a + b < 1$:

1. Sort indices by $|g_i|$ descending.
2. Let $A$ = top $a \cdot n$ indices (the *large-gradient* set, kept exactly).
3. Sample $b \cdot n$ indices uniformly without replacement from the rest; call it $B$.
4. Scale the gradients and Hessians of rows in $B$ by $(1-a)/b$.
5. Build the histogram on $A \cup B$ and find the best split as usual.

The amplification factor $(1-a)/b$ corrects for the subsampling so the expected histogram still matches the full-data histogram (Section 2 below).

### Default settings

LightGBM defaults are roughly $a = 0.2$, $b = 0.1$, which means each iteration touches 30% of the rows — a $\sim 3 \times$ speed-up at modest variance cost.


## 2. Unbiasedness of GOSS — sketch of the proof

Let $\hat{V}_j(d, \tau)$ be the GOSS estimator of the split gain for feature $j$ at threshold $\tau$, and let $V_j(d, \tau)$ be the true gain on the full dataset.

**Claim:** $\mathbb{E}[\hat{V}_j(d, \tau)] = V_j(d, \tau)$.

**Proof sketch.** The gain identity from Week 1 is

$$
V_j(d, \tau) = \frac{1}{2} \left[ \frac{(\sum_{i \in I_L} g_i)^2}{|I_L|} + \frac{(\sum_{i \in I_R} g_i)^2}{|I_R|} \right] - \text{const},
$$

(temporarily ignoring $\lambda$ and using unit Hessians for clarity). The numerator on each side is $G_L = \sum_{i \in I_L} g_i$ and similarly for $G_R$. Decompose $I_L = (I_L \cap A) \cup (I_L \cap B^c \cap B_{\text{pool}})$:

$$
\sum_{i \in I_L} g_i = \sum_{i \in I_L \cap A} g_i + \sum_{i \in I_L \cap B_{\text{pool}}} g_i.
$$

GOSS replaces the second term with its scaled subsample: $\frac{1-a}{b} \sum_{i \in I_L \cap B} g_i$, where $B$ is a uniform sample of size $b \cdot n$ from the pool $B_{\text{pool}}$ of size $(1-a) n$. Under uniform sampling without replacement,

$$
\mathbb{E}\!\left[ \sum_{i \in I_L \cap B} g_i \right] = \frac{b \cdot n}{(1-a) n} \cdot \sum_{i \in I_L \cap B_{\text{pool}}} g_i = \frac{b}{1-a} \cdot \sum_{i \in I_L \cap B_{\text{pool}}} g_i.
$$

Multiplying by the amplification factor $(1-a)/b$:

$$
\mathbb{E}\!\left[ \frac{1-a}{b} \sum_{i \in I_L \cap B} g_i \right] = \sum_{i \in I_L \cap B_{\text{pool}}} g_i.
$$

Therefore the GOSS estimator of $G_L$ — large-gradient sum plus amplified small-gradient subsample — has the correct expectation, and the gain estimator inherits unbiasedness (modulo the small bias from the squared numerator, which Ke et al. bound at $O(\sqrt{(\ln n)/n})$ with high probability). $\blacksquare$

**Take-away:** GOSS is not heuristic. It trades a $O(\sqrt{(\ln n)/n})$ approximation error for an $O(1/(a + b))$ speed-up — almost always a winning trade.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))

from gradient_forge.lightgbm_internals.goss import GradientBasedOneSideSampling
from gradient_forge.lightgbm_internals.efb import ExclusiveFeatureBundling
from gradient_forge.lightgbm_internals import LightGBMTrainer
from gradient_forge.xgboost_internals import XGBoostTrainer
from gradient_forge.data.loaders import load_synthetic_classification
from gradient_forge.utils import Stopwatch, MemoryProfiler, seed_everything
seed_everything(42)


### 2.1 GOSS in action

Verify the unbiasedness empirically: sample many GOSS subsets from the same gradient vector and check that the mean of the estimated full-data gradient sum equals the true sum.


In [ ]:
rng = np.random.default_rng(0)
n = 5_000
g_true = rng.normal(size=n)
h_true = np.ones_like(g_true)
true_sum = float(g_true.sum())

# Run GOSS many times; each time, estimate the full-data gradient sum from the subsample.
estimates = []
for trial in range(500):
    sampler = GradientBasedOneSideSampling(top_rate=0.2, other_rate=0.1, random_state=trial)
    idx, g_s, h_s = sampler.sample(g_true, h_true)
    # The amplified gradients in `g_s` already encode the correction; their sum estimates the full sum.
    estimates.append(float(g_s.sum()))

print(f"True full-data Σ g_i           : {true_sum:>10.4f}")
print(f"Mean GOSS estimate (500 runs)  : {np.mean(estimates):>10.4f}")
print(f"Standard error of the estimate : {np.std(estimates):>10.4f}")
print(f"Relative bias                  : {abs(np.mean(estimates) - true_sum) / abs(true_sum):.4f}")


In [ ]:
# Visualize which rows GOSS keeps vs. discards.
g = np.linspace(-5, 5, 1000)
h = np.ones_like(g)
sampler = GradientBasedOneSideSampling(top_rate=0.2, other_rate=0.1, random_state=0)
idx, g_sampled, h_sampled = sampler.sample(g, h)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(np.abs(g), bins=40, alpha=0.5, label="all |g|")
axes[0].hist(np.abs(g[idx]), bins=40, alpha=0.7, color="C3", label="GOSS-selected")
axes[0].set_title("GOSS keeps large-|g| rows, subsamples the small-|g| pool")
axes[0].set_xlabel("|gradient|"); axes[0].legend()

axes[1].scatter(g[idx], h_sampled, s=10, alpha=0.6)
axes[1].axhline(1.0, color="grey", linestyle=":")
axes[1].axhline((1 - 0.2) / 0.1, color="C3", linestyle="--",
                label=f"amplification (1-a)/b = {(1-0.2)/0.1:.1f}")
axes[1].set_xlabel("gradient"); axes[1].set_ylabel("post-scaling Hessian")
axes[1].set_title("Small-|g| rows are amplified by (1-a)/b"); axes[1].legend()
plt.tight_layout(); plt.show()


## 3. EFB — Exclusive Feature Bundling

In sparse high-dimensional data (one-hot encodings, click streams, bag-of-words), many features are **mutually exclusive** — they rarely take non-zero values simultaneously. Bundling them into a single dense feature reduces the effective dimensionality $d$ in the per-iteration histogram cost.

### Reduction to graph coloring

Build the **conflict graph** $G = (V, E)$:

- Vertices $V$ = features.
- Edge $(i, j) \in E$ if features $i$ and $j$ are simultaneously non-zero on more than $K$ rows (where $K$ is a tolerance).

Bundling features into the minimum number of dense columns is **graph coloring**: assign each vertex a color (bundle ID) such that adjacent vertices get different colors. Graph coloring is NP-hard in general, but a **greedy heuristic** is sufficient in practice:

1. Sort vertices by degree (descending).
2. For each vertex, assign the first color (bundle) that doesn't conflict with its neighbors.

This $O(d^2)$ procedure is the entire EFB algorithm.

### Within-bundle merging

Once a set of features shares a bundle, we need to keep them distinguishable inside the dense column. Shift each feature's values by a per-feature offset so its range does not overlap with the others' ranges. The decoded value still identifies which original feature contributed it.

### How much does this save?

For one-hot encoded categoricals with $K$ levels, EFB collapses $K$ columns into 1 — a $K\times$ reduction in $d$. Combined with GOSS's row reduction, LightGBM achieves order-of-magnitude speed-ups on sparse high-dimensional tabular data.


In [ ]:
# Build a synthetic example with three exclusive feature groups.
rng = np.random.default_rng(0)
n = 500
X = np.zeros((n, 10), dtype=np.float64)

# Group 1 (cols 0..2): one-hot over 3 levels.
X[np.arange(n), rng.integers(0, 3, size=n)] = 1.0
# Group 2 (cols 3..5): another one-hot over 3 levels.
X[np.arange(n), 3 + rng.integers(0, 3, size=n)] = 1.0
# Group 3 (cols 6..8): a third one-hot.
X[np.arange(n), 6 + rng.integers(0, 3, size=n)] = 1.0
# Col 9: dense Gaussian (must NOT be bundled with anything).
X[:, 9] = rng.normal(size=n)

efb = ExclusiveFeatureBundling(max_conflict_rate=0.0).fit(X)
print(f"Original features : {X.shape[1]}")
print(f"After EFB         : {len(efb.bundles_)} bundles")
print()
for k, b in enumerate(efb.bundles_):
    print(f"  bundle {k}: features {b.feature_ids}  (offsets {[f'{o:.2f}' for o in b.offsets]})")


In [ ]:
# Visualize the transformation.
X_bundled = efb.transform(X)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].imshow(X[:50].T, aspect="auto", cmap="Blues")
axes[0].set_title(f"Before EFB — {X.shape[1]} sparse columns")
axes[0].set_xlabel("row"); axes[0].set_ylabel("feature")

axes[1].imshow(X_bundled[:50].T, aspect="auto", cmap="Blues")
axes[1].set_title(f"After EFB — {X_bundled.shape[1]} dense bundles")
axes[1].set_xlabel("row"); axes[1].set_ylabel("bundle")
plt.tight_layout(); plt.show()


## 4. Leaf-wise vs. level-wise growth

| Strategy | How it grows | Pros | Cons |
|----------|--------------|------|------|
| **Level-wise** (XGBoost default) | Splits *every* leaf at depth $d$ before any leaf at depth $d+1$. | Tree shape is regular; parallelization across leaves is easy. | Wastes capacity on low-gain leaves. |
| **Leaf-wise** (LightGBM default) | At each step, splits the *single* leaf with the highest gain regardless of depth. | Each split is the best available — converges in fewer iterations. | Trees grow asymmetrically deep; severe overfitting risk if `num_leaves` is high relative to data size. |

### Regularization implication

Leaf-wise growth converges faster but is **strictly more prone to overfitting**. The mitigations are:

- **`num_leaves` ≤ 64** for small/medium datasets.
- **`min_child_samples` ≥ 20** to prevent splits on tiny populations.
- **Always** use early stopping.
- Consider `max_depth` as a hard cap even when growing leaf-wise.

LightGBM defaults are *aggressive*: `num_leaves = 31` looks small but corresponds to depth ≈ 5 in a balanced tree — and can be deeper on the highest-gain branch.


## 5. Head-to-head benchmark — XGBoost vs. LightGBM

Run both libraries on the same 50k-row dataset with identical learning rate and tree count, then measure:

- **Training time** (median over 3 runs)
- **Peak memory** (via `tracemalloc`)
- **Prediction latency** per 10k rows
- **Test-set AUC**


In [ ]:
X, y = load_synthetic_classification(n_samples=50_000, n_features=40, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

results = []
common = dict(task="binary", n_estimators=400, learning_rate=0.05, random_state=42)

# --- XGBoost ---
with Stopwatch() as sw_xgb_train, MemoryProfiler() as mp_xgb:
    xgb_model = XGBoostTrainer(max_depth=6, **common).fit(
        X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30
    )
with Stopwatch() as sw_xgb_pred:
    p_xgb = xgb_model.predict_proba(X_test)[:, 1]
results.append({
    "model": "XGBoost", "train_s": sw_xgb_train.seconds,
    "peak_MB": mp_xgb.peak_mb, "pred_s": sw_xgb_pred.seconds,
    "AUC": float(roc_auc_score(y_test, p_xgb)),
})

# --- LightGBM ---
with Stopwatch() as sw_lgb_train, MemoryProfiler() as mp_lgb:
    lgb_model = LightGBMTrainer(num_leaves=63, **common).fit(
        X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30
    )
with Stopwatch() as sw_lgb_pred:
    p_lgb = lgb_model.predict_proba(X_test)[:, 1]
results.append({
    "model": "LightGBM", "train_s": sw_lgb_train.seconds,
    "peak_MB": mp_lgb.peak_mb, "pred_s": sw_lgb_pred.seconds,
    "AUC": float(roc_auc_score(y_test, p_lgb)),
})

df_bench = pd.DataFrame(results)
df_bench


In [ ]:
# Pareto plot: training time vs. AUC.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for _, row in df_bench.iterrows():
    axes[0].scatter(row["train_s"], row["AUC"], s=140)
    axes[0].annotate(row["model"], (row["train_s"], row["AUC"]),
                     xytext=(10, 4), textcoords="offset points", fontsize=11)
axes[0].set_xlabel("training time (s)"); axes[0].set_ylabel("test AUC")
axes[0].set_title("Speed–accuracy Pareto frontier"); axes[0].grid(alpha=0.3)

models = df_bench["model"].tolist()
axes[1].bar(models, df_bench["peak_MB"], color=["C0", "C3"])
axes[1].set_ylabel("peak memory (MB)"); axes[1].set_title("Training-time memory footprint")
axes[1].grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()


## 6. Overfitting demonstration

Crank `num_leaves` up to 511 with `min_child_samples = 2` and no early stopping. The leaf-wise growth pursues the highest-gain split at every step — exactly the recipe for fitting noise.


In [ ]:
# Subsample to make overfitting more visible.
X_sub = X_train[:5_000]; y_sub = y_train[:5_000]

reckless = LightGBMTrainer(
    task="binary", n_estimators=300, learning_rate=0.05,
    num_leaves=511, min_child_samples=2,
    feature_fraction=1.0, bagging_fraction=1.0,
    random_state=42,
).fit(X_sub, y_sub, eval_set=(X_test, y_test), early_stopping_rounds=None)

careful = LightGBMTrainer(
    task="binary", n_estimators=300, learning_rate=0.05,
    num_leaves=31, min_child_samples=20,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    random_state=42,
).fit(X_sub, y_sub, eval_set=(X_test, y_test), early_stopping_rounds=None)

print(f"Reckless (num_leaves=511, min_child=2)  : train AUC = "
      f"{roc_auc_score(y_sub, reckless.predict_proba(X_sub)[:, 1]):.4f}   "
      f"test AUC = {roc_auc_score(y_test, reckless.predict_proba(X_test)[:, 1]):.4f}")
print(f"Careful  (num_leaves=31,  min_child=20) : train AUC = "
      f"{roc_auc_score(y_sub, careful.predict_proba(X_sub)[:, 1]):.4f}   "
      f"test AUC = {roc_auc_score(y_test, careful.predict_proba(X_test)[:, 1]):.4f}")
print("\nThe reckless model fits training noise; the careful model generalizes.")


## 7. Sparse data benefit — when EFB pays off

Construct a high-cardinality one-hot encoded dataset to see EFB's impact in a setting where it matters most.


In [ ]:
rng = np.random.default_rng(0)
n_rows = 20_000
n_categoricals = 5     # number of categorical features
n_levels = 30          # levels per categorical (each gets a one-hot column)

# Build one-hot encoded data.
ohe_blocks = []
for _ in range(n_categoricals):
    block = np.zeros((n_rows, n_levels), dtype=np.float64)
    block[np.arange(n_rows), rng.integers(0, n_levels, size=n_rows)] = 1.0
    ohe_blocks.append(block)
X_sparse = np.hstack(ohe_blocks)
# Manufacture a label correlated with a few specific levels.
y_sparse = ((X_sparse[:, 0] + X_sparse[:, n_levels + 5] + X_sparse[:, 3 * n_levels + 10]) > 0).astype(int)

print(f"Sparse dataset: {X_sparse.shape[0]} rows × {X_sparse.shape[1]} columns")
print(f"Sparsity     : {(X_sparse == 0).mean():.2%} zeros")

with Stopwatch() as sw_no_efb:
    LightGBMTrainer(task="binary", n_estimators=200, num_leaves=31,
                    extra_params={"enable_bundle": False},
                    random_state=42).fit(X_sparse, y_sparse)
with Stopwatch() as sw_with_efb:
    LightGBMTrainer(task="binary", n_estimators=200, num_leaves=31,
                    extra_params={"enable_bundle": True},
                    random_state=42).fit(X_sparse, y_sparse)
print(f"\nLightGBM training time:")
print(f"  EFB disabled : {sw_no_efb.seconds:.2f}s")
print(f"  EFB enabled  : {sw_with_efb.seconds:.2f}s   "
      f"({sw_no_efb.seconds / max(sw_with_efb.seconds, 1e-6):.2f}× speed-up)")


## 8. Exercises

1. **GOSS variance budget.** Vary $a + b$ from 0.1 to 0.9 and measure the test-AUC variance over 20 random seeds. At what total sampling rate does variance overwhelm the speed-up?
2. **Conflict tolerance in EFB.** Increase `max_conflict_rate` from 0 to 0.1. How many bundles are produced? Does training AUC suffer?
3. **`num_leaves` sweep.** For a fixed early-stopping budget, plot test AUC as `num_leaves` ranges over $\{15, 31, 63, 127, 255, 511\}$. Where is the optimum on your dataset?
4. **Replicate Pareto frontier on your data.** Repeat Section 5 with a dataset of your choice and confirm — or refute — the conventional wisdom that LightGBM dominates on speed.

## Takeaways

- **GOSS** is a *provably unbiased* gradient-sum estimator with $O((a+b)^{-1})$ speed-up at $O(\sqrt{(\ln n)/n})$ variance cost.
- **EFB** turns sparse one-hot columns into dense bundles via greedy graph coloring — the dominant savings on click-stream and recommendation data.
- **Leaf-wise growth** is regularization-sensitive; treat `num_leaves` as a *hyperparameter to tune*, not a default to accept.
- LightGBM typically wins on speed and matches XGBoost on accuracy when both are tuned.

> **Next week:** CatBoost addresses a *statistical* (not computational) flaw — target leakage in categorical encodings and prediction shift in gradient estimation.
